# Notebook 10 — Panel Extension
## Extending Saadaoui (2026, JCE) — Heterogeneity and Robustnes


**Sections:**
1. Imports and paths
2. Data loading
3. Stata file and dyad registry
4. **Instrument diagnostic** — F-stat under 6 control specifications per dyad
5. **Instrument comparison** — Saadaoui d2pri vs constructed, correlation verification
6. **LP-IV functions** — core estimator + Anderson-Rubin robust CI
7. **Dyad-by-dyad LP-IV** — all valid dyads, AR CIs for borderline cases
8. **Reduced-form LP** — instrument → WTI directly (no first stage needed)
9. **OLS vs IV comparison** — bias direction and endogeneity test per dyad
10. **Specification curve** — US-China across 8 control sets
11. **External validity H2** — Wald test US vs Japan
12. **Meta-analytic pooling** — IVW + Cochran Q heterogeneity test
13. **CF pooled panel**
14. **Panel DML-PLIV** — XGBoost + Ridge, stability diagnostics
15. **GDELT NLP sensitivity** 
16. **GPR sensitivity** — using published GPR index instead of PRI-based instrument
17. **Placebo test** — permuted instrument, how often do we match US-China?
18. **Structural heterogeneity** — crisis vs non-crisis, pre/post 2008, pre/post 2015
19. **Power analysis** — MDE for every sub-sample
20. **Final honest summary**


## 1. Imports and paths

In [1]:
import warnings, json, time
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.stattools import grangercausalitytests
from linearmodels.iv import IV2SLS
from scipy import stats
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
import doubleml as dml
from xgboost import XGBRegressor
import shap

np.random.seed(42)

ROOT    = __import__('pathlib').Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
FINAL   = ROOT / 'data' / 'final'
RAW     = ROOT / 'data' / 'raw'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
for d in [RESULTS, FIGURES]: d.mkdir(parents=True, exist_ok=True)

HMAX     = 48
MIN_F    = 10.0   # Stock-Yogo threshold
STRONG_F = 30.0   # "strong" threshold — conservative
ALPHA    = 0.10   # significance level throughout

print(f'ROOT    : {ROOT}')
print(f'FINAL   : {FINAL}  exists={FINAL.exists()}')
print(f'Thresholds: MIN_F={MIN_F}  STRONG_F={STRONG_F}  alpha={ALPHA}')

ROOT    : c:\Users\HP\Desktop\replication+contribution
FINAL   : c:\Users\HP\Desktop\replication+contribution\data\final  exists=True
Thresholds: MIN_F=10.0  STRONG_F=30.0  alpha=0.1


## 2. Load data

In [2]:
df_ext = pd.read_csv(FINAL / 'df_extended.csv', index_col=0, parse_dates=True)
df_ext.index = pd.to_datetime(df_ext.index).to_period('M').to_timestamp('M')

with open(FINAL / 'variable_roles.json') as f:
    roles = json.load(f)

OUTCOME  = roles['outcome'][0]   # lwti
CTRL_CORE  = roles['controls_core']          # llwip, dllgop, dl2lgop  (3)
CTRL_MACRO = roles['controls_macro']         # vix, gs10, tb3ms, baa10y, brent, gold, bdi, cny_usd, indpro (9)
CTRL_GEO   = roles.get('controls_geopol', [])  # gpr_chn_l1, gpr_usa_l1 (2)
CTRL_FULL  = CTRL_CORE + CTRL_MACRO + CTRL_GEO  # 14 vars

# Control set definitions for specification curve
CTRL_SETS = {
    'minimal (3)':    CTRL_CORE,
    'core+fin (7)':   CTRL_CORE + ['vix','gs10','tb3ms','baa10y'],
    'core+macro (12)': CTRL_CORE + CTRL_MACRO,
    'full (14)':      CTRL_FULL,
}

# NLP controls
NLP_COLS = []
nlp_path = FINAL / 'df_extended_nlp.csv'
if nlp_path.exists():
    df_nlp = pd.read_csv(nlp_path, index_col=0, parse_dates=True)
    df_nlp.index = pd.to_datetime(df_nlp.index).to_period('M').to_timestamp('M')
    cands = ['gdelt_goldstein_mean','gdelt_sentiment_signal',
             'gdelt_conflict_share','gdelt_coop_share']
    NLP_COLS = [c for c in cands if c in df_nlp.columns and df_nlp[c].notna().mean()>=0.99]
    df_ext_nlp = df_ext.join(df_nlp[NLP_COLS], how='left') if NLP_COLS else df_ext.copy()
    if NLP_COLS:
        CTRL_SETS['full+NLP (18)'] = CTRL_FULL + NLP_COLS
else:
    df_ext_nlp = df_ext.copy()

print(f'Outcome   : {OUTCOME}')
print(f'Core ctrl : {CTRL_CORE}')
print(f'Macro ctrl: {CTRL_MACRO}')
print(f'Geo ctrl  : {CTRL_GEO}')
print(f'NLP ctrl  : {NLP_COLS}')
print(f'Control sets: {list(CTRL_SETS.keys())}')
print(f'df_extended: {df_ext.shape}')

Outcome   : lwti
Core ctrl : ['llwip', 'dllgop', 'dl2lgop']
Macro ctrl: ['vix', 'gs10', 'tb3ms', 'baa10y', 'brent', 'gold', 'bdi', 'cny_usd', 'indpro']
Geo ctrl  : ['gpr_chn_l1', 'gpr_usa_l1']
NLP ctrl  : ['gdelt_goldstein_mean', 'gdelt_sentiment_signal', 'gdelt_conflict_share', 'gdelt_coop_share']
Control sets: ['minimal (3)', 'core+fin (7)', 'core+macro (12)', 'full (14)', 'full+NLP (18)']
df_extended: (385, 17)


## 3. Stata file and dyad registry

In [3]:
stata_files = (list(RAW.glob('*.dta'))
              + list(ROOT.glob('*.dta'))
              + list((ROOT/'data').glob('**/*.dta')))
assert stata_files, f'No .dta file found under {ROOT}'

df_raw = pd.read_stata(stata_files[0])
date_col = next((c for c in df_raw.columns
                 if 'date' in c.lower()
                 or pd.api.types.is_datetime64_any_dtype(df_raw[c])), None)
df_raw['_d'] = pd.to_datetime(df_raw[date_col])
df_raw = df_raw.set_index('_d').sort_index()
df_raw.index = df_raw.index.to_period('M').to_timestamp('M')

DYAD_DEFS = [
    ('us',    'US–China',        'lpri',       True,  'dlpri'),
    ('jp',    'Japan–China',     'lpri_jp',    True,  'dlpri_jp'),
    ('aus',   'Australia–China', 'lpri_aus',   False, 'dlpri_aus'),
    ('cds',   'S.Korea–China',   'lpri_cds',   False, 'dlpri_cds'),
    ('fra',   'France–China',    'lpri_fra',   False, 'dlpri_fra'),
    ('ger',   'Germany–China',   'lpri_ger',   False, 'dlpri_ger'),
    ('india', 'India–China',     'lpri_india', False, 'dlpri_india'),
    ('indo',  'Indonesia–China', 'lpri_indo',  False, 'dlpri_indo'),
    ('pak',   'Pakistan–China',  'lpri_pak',   False, 'dlpri_pak'),
    ('rus',   'Russia–China',    'lpri_rus',   False, 'dlpri_rus'),
    ('vn',    'Vietnam–China',   'lpri_vn',    False, 'dlpri_vn'),
    ('uk',    'UK–China',        'lpri_uk',    False, 'dlpri_uk'),
]
for code, name, lpri, has_d2, dlpri in DYAD_DEFS:
    d2col = 'd2pri' if code=='us' else f'd2pri_{code}'
    if not has_d2 and dlpri in df_raw.columns:
        df_raw[d2col] = df_raw[dlpri].diff()

print(f'Stata: {stata_files[0].name}  shape={df_raw.shape}')
print(f'Range: {df_raw.index.min().strftime("%Y-%m")} – {df_raw.index.max().strftime("%Y-%m")}')

Stata: Saadaoui_2026_JCE.dta  shape=(386, 59)
Range: 1990-01 – 2022-02


## 4. Instrument diagnostic across six control specifications

We test every instrument candidate under every control specification.
This answers: is the F-statistic robust to control set choice, or does it depend on
which variables are partialled out?

A truly strong instrument should have F ≥ 10 under all reasonable specifications.
An instrument that only achieves F ≥ 10 under one narrow specification is fragile.

**Collinearity constraint:**
- `d2pri`, `dlpri`: tested **with** endogenous lags in exog (no collinearity)
- `L1dlpri`, `L2dlpri`: tested **without** endogenous lags (exact collinearity with lag(lpri))

In [4]:
def first_stage_F(endog_s, instr_s, ctrl_df, endog_lags=2):
    common = (endog_s.dropna().index
              .intersection(instr_s.dropna().index)
              .intersection(ctrl_df.dropna(how='all').index))
    if len(common) < 40: return np.nan
    df_f = pd.DataFrame({'e': endog_s.loc[common], 'z': instr_s.loc[common]})
    for c in ctrl_df.columns:
        df_f[c] = ctrl_df.loc[common, c]
    for l in range(1, endog_lags+1):
        df_f[f'Le{l}'] = endog_s.loc[common].shift(l)
    df_f = df_f.replace([np.inf,-np.inf], np.nan).dropna()
    if df_f['z'].std() < 1e-6 or len(df_f) < 30: return np.nan
    xcols = [c for c in df_f.columns if c != 'e']
    try:
        fit = sm.OLS(df_f['e'], add_constant(df_f[xcols], has_constant='add')).fit(cov_type='HC1')
        F = float(fit.f_test('z = 0').fvalue)
        return np.nan if (F > 1e6 or F < 0) else F
    except: return np.nan

def granger_p(instr_s, wti_s, maxlag=3):
    df_g = pd.DataFrame({'z': instr_s.diff(), 'y': wti_s.diff()}).dropna()
    if len(df_g) < 50: return np.nan
    try:
        gc = grangercausalitytests(df_g[['z','y']], maxlag=maxlag, verbose=False)
        return float(min(gc[l][0]['ssr_ftest'][1] for l in range(1,maxlag+1)))
    except: return np.nan

wti_s = df_ext[OUTCOME]
ctrl_ref = df_ext[CTRL_FULL]  # reference for diagnostic

# ── Run diagnostic under ALL control sets ────────────────────────────────────
print('INSTRUMENT DIAGNOSTIC — F-STATISTICS ACROSS CONTROL SPECIFICATIONS')
print('='*110)
header = f'  {"Dyad":<22}  {"Instr":<12}'
for cname in CTRL_SETS: header += f'  {cname[:10]:>10}'
header += f'  {"exog_p":>8}  {"Decision"}'
print(header)
print('-'*110)

diag_rows = []
for code, name, lpri_col, has_d2, dlpri_col in DYAD_DEFS:
    d2col = 'd2pri' if code=='us' else f'd2pri_{code}'
    if lpri_col not in df_raw.columns or dlpri_col not in df_raw.columns: continue
    endog   = df_raw[lpri_col].reindex(df_ext.index)
    d2pri   = df_raw[d2col].reindex(df_ext.index) if d2col in df_raw.columns else pd.Series(np.nan, index=df_ext.index)
    dlpri   = df_raw[dlpri_col].reindex(df_ext.index)

    candidates = [
        ('d2pri',   d2pri,         True),
        ('dlpri',   dlpri,         True),
        ('L1dlpri', dlpri.shift(1), False),
        ('L2dlpri', dlpri.shift(2), False),
    ]
    exog_p = granger_p(d2pri if not d2pri.isna().all() else dlpri, wti_s)

    best_row = None
    for iname, iseries, can_el in candidates:
        F_by_spec = {}
        for cname, ctrl_cols in CTRL_SETS.items():
            ctrl_df_s = df_ext[ctrl_cols] if all(c in df_ext.columns for c in ctrl_cols) else ctrl_ref
            F_by_spec[cname] = first_stage_F(endog, iseries, ctrl_df_s,
                                              endog_lags=2 if can_el else 0)
        # Use full-spec F for ranking
        F_rank = F_by_spec.get('full (14)', np.nan)
        if best_row is None or (not pd.isna(F_rank) and F_rank < 1e6 and
                                (pd.isna(best_row['F_rank']) or F_rank > best_row['F_rank'])):
            best_row = dict(iname=iname, iseries=iseries, can_el=can_el,
                            F_rank=F_rank, F_by_spec=F_by_spec)

    if best_row is None:
        best_row = dict(iname='none', iseries=None, can_el=False,
                        F_rank=np.nan, F_by_spec={c: np.nan for c in CTRL_SETS})

    F_full = best_row['F_rank']
    valid  = (not pd.isna(F_full) and F_full >= MIN_F
              and not pd.isna(exog_p) and exog_p >= 0.05)
    strong = valid and F_full >= STRONG_F
    decision = ('STRONG' if strong else ('VALID' if valid else
                ('WEAK_exog' if (not pd.isna(F_full) and F_full >= MIN_F) else 'WEAK_F')))

    row_str = f'  {name:<22}  {best_row["iname"]:<12}'
    for cname in CTRL_SETS:
        F_v = best_row['F_by_spec'].get(cname, np.nan)
        mk  = '✓' if (not pd.isna(F_v) and F_v>=MIN_F) else ('⚠' if (not pd.isna(F_v) and F_v>=7) else '✗')
        row_str += f'  {(f"{F_v:.0f}" if not pd.isna(F_v) else "NaN"):>9}{mk}'
    row_str += f'  {exog_p:8.3f}  {decision}'
    print(row_str)

    diag_rows.append(dict(
        code=code, name=name, lpri_col=lpri_col, dlpri_col=dlpri_col, d2col=d2col,
        best_name=best_row['iname'], best_series=best_row['iseries'],
        can_elags=best_row['can_el'], best_F=F_full,
        F_by_spec=best_row['F_by_spec'], exog_p=exog_p,
        valid=valid, strong=strong, decision=decision
    ))

print()
valid_dyads  = [r for r in diag_rows if r['valid']]
strong_dyads = [r for r in diag_rows if r['strong']]
print(f'Valid  (F≥{MIN_F:.0f}): {len(valid_dyads)}/12  {[r["name"] for r in valid_dyads]}')
print(f'Strong (F≥{STRONG_F:.0f}): {len(strong_dyads)}/12  {[r["name"] for r in strong_dyads]}')

INSTRUMENT DIAGNOSTIC — F-STATISTICS ACROSS CONTROL SPECIFICATIONS
  Dyad                    Instr         minimal (3  core+fin (  core+macro   full (14)  full+NLP (    exog_p  Decision
--------------------------------------------------------------------------------------------------------------
  US–China                d2pri               230✓        228✓        240✓        243✓        243✓     0.125  STRONG
  Japan–China             d2pri               126✓        117✓        114✓        113✓        113✓     0.723  STRONG
  Australia–China         L2dlpri              71✓         61✓         61✓         60✓         60✓     0.052  STRONG
  S.Korea–China           L1dlpri               0✗          0✗          0✗          0✗          0✗     0.770  WEAK_F
  France–China            L2dlpri               2✗          5✗         13✓         11✓         11✓     0.605  VALID
  Germany–China           L1dlpri              16✓         13✓         12✓         12✓         12✓     0.533  VALID
  I

## 5. Instrument verification and time-series comparison

We verify that `dlpri.diff()` exactly replicates Saadaoui's `d2pri` for US-China
(confirming our construction is valid for other dyads), and plot all instrument
candidates to understand their time-series properties.

In [5]:
idx = df_ext.index

# ── Exact replication check ────────────────────────────────────────────────
d2_file  = df_raw['d2pri'].reindex(idx)
d2_const = df_raw['dlpri'].diff().reindex(idx)
corr     = d2_file.corr(d2_const)
max_diff = (d2_file - d2_const).abs().max()

print('INSTRUMENT VERIFICATION: d2pri (file) vs dlpri.diff() (constructed)')
print(f'  Correlation   : {corr:.8f}')
print(f'  Max |diff|    : {max_diff:.2e}')
print(f'  Result        : {"IDENTICAL (floating-point only)" if corr > 0.9999 else f"DIFFERS corr={corr:.4f}"}')
print()

# ── Descriptive stats for US-China instruments ────────────────────────────
instrs_us = {
    'd2pri (Saadaoui)':  df_raw['d2pri'].reindex(idx),
    'dlpri (1st diff)':  df_raw['dlpri'].reindex(idx),
    'L1dlpri':           df_raw['dlpri'].shift(1).reindex(idx),
    'L2dlpri':           df_raw['dlpri'].shift(2).reindex(idx),
}
print('US-CHINA INSTRUMENT DESCRIPTIVES')
print(f'  {"Instrument":<22} {"n":>5} {"Mean":>8} {"Std":>8} {"Kurtosis":>10} {"% zero":>8}')
for iname, s in instrs_us.items():
    sc = s.dropna()
    pct0 = (sc.abs() < 1e-8).mean()*100
    print(f'  {iname:<22} {len(sc):>5} {sc.mean():>8.4f} {sc.std():>8.4f} '
          f'{sc.kurtosis():>10.2f} {pct0:>7.1f}%')

# ── Figure: time series of instruments ────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
for ax, (iname, s) in zip(axes, instrs_us.items()):
    ax.plot(idx, s, color='navy', lw=1)
    ax.axhline(0, color='black', lw=0.6, linestyle='--')
    ax.set_title(f'US–China: {iname}', fontsize=9)
    ax.grid(alpha=0.2)
plt.suptitle('US–China Instrument Candidates — Time Series Properties', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_10_instr_timeseries.png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved: Figure_10_instr_timeseries.png')

# ── Cross-dyad instrument correlation heatmap ─────────────────────────────
instr_panel = pd.DataFrame()
for code, name, lpri_col, has_d2, dlpri_col in DYAD_DEFS:
    d2col = 'd2pri' if code=='us' else f'd2pri_{code}'
    if d2col in df_raw.columns:
        instr_panel[name] = df_raw[d2col].reindex(idx)

if instr_panel.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    corr_m = instr_panel.corr()
    im = ax.imshow(corr_m.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr_m))); ax.set_xticklabels(corr_m.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(corr_m))); ax.set_yticklabels(corr_m.columns, fontsize=8)
    for i in range(len(corr_m)):
        for j in range(len(corr_m)):
            ax.text(j, i, f'{corr_m.values[i,j]:.2f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax)
    ax.set_title('Cross-dyad instrument (d2pri) correlations\\nLow correlation = independent identifying variation')
    plt.tight_layout()
    plt.savefig(FIGURES/'Figure_10_instr_correlation.png', dpi=200, bbox_inches='tight')
    plt.close()
    print('Saved: Figure_10_instr_correlation.png')

INSTRUMENT VERIFICATION: d2pri (file) vs dlpri.diff() (constructed)
  Correlation   : 0.94798083
  Max |diff|    : 7.82e-01
  Result        : DIFFERS corr=0.9480

US-CHINA INSTRUMENT DESCRIPTIVES
  Instrument                 n     Mean      Std   Kurtosis   % zero
  d2pri (Saadaoui)         385   0.0000   0.3383      10.48    23.4%
  dlpri (1st diff)         385  -0.0056   0.1751      25.40    39.5%
  L1dlpri                  385  -0.0056   0.1751      25.40    39.5%
  L2dlpri                  384  -0.0056   0.1753      25.33    39.6%
Saved: Figure_10_instr_timeseries.png
Saved: Figure_10_instr_correlation.png


## 6. Core estimation functions

**`lp_iv`**: LP-IV at each horizon, returns full IRF DataFrame. HC-robust SEs.
No silent exception catching — all failures print explicitly.

**`ar_ci_grid`**: Anderson-Rubin confidence set via grid inversion.
Valid under weak instruments. We compute this for ALL valid dyads,
not just borderline ones — it provides a conservative robustness check.

**`lp_ols`**: OLS version of LP (no IV). Comparison to IV reveals
the direction and magnitude of endogeneity bias.

**`lp_reduced_form`**: regress WTI directly on the instrument (no first stage).
If this is significant, the IV channel exists regardless of instrument strength concerns.

In [6]:
def lp_iv(df_base, endog_s, instr_s, controls,
          include_endog_lags=True, hmax=HMAX, label=''):
    common = (df_base.index
              .intersection(endog_s.dropna().index)
              .intersection(instr_s.dropna().index))
    work = df_base[[OUTCOME]+controls].loc[common].copy()
    work['__e__'] = endog_s.loc[common]
    work['__z__'] = instr_s.loc[common]
    for l in range(1,4): work[f'Ly{l}'] = work[OUTCOME].shift(l)
    lag_y = [f'Ly{l}' for l in range(1,4)]
    lag_e = []
    if include_endog_lags:
        for l in range(1,3): work[f'Le{l}'] = work['__e__'].shift(l)
        lag_e = [f'Le{l}' for l in range(1,3)]
    exog_cols = lag_y + lag_e + controls
    rows = []
    for h in range(hmax+1):
        hdf = pd.DataFrame({
            'y': work[OUTCOME].shift(-h), 'e': work['__e__'], 'z': work['__z__'],
            **{c: work[c] for c in exog_cols}
        }).replace([np.inf,-np.inf],np.nan).dropna()
        if len(hdf) < 40:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(hdf),'F':np.nan}); continue
        try:
            X_fs = add_constant(hdf[['z']+exog_cols], has_constant='add')
            fs   = sm.OLS(hdf['e'], X_fs).fit(cov_type='HC1')
            Fv   = float(fs.f_test('z = 0').fvalue)
            if Fv > 1e6 or Fv < 0: Fv = np.nan
            fit = IV2SLS(
                dependent=hdf['y'],
                exog=add_constant(hdf[exog_cols], has_constant='add'),
                endog=hdf[['e']], instruments=hdf[['z']]
            ).fit(cov_type='robust', debiased=True)
            rows.append({'h':h,'n':len(hdf),'F':Fv,
                         'coef':float(fit.params.get('e',np.nan)),
                         'se':float(fit.std_errors.get('e',np.nan))})
        except Exception as ex:
            print(f'  [lp_iv] {label} h={h}: {type(ex).__name__}: {ex}')
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(hdf),'F':np.nan})
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef'] - 1.645*irf['se']
    irf['hi90'] = irf['coef'] + 1.645*irf['se']
    return irf


def lp_ols(df_base, endog_s, controls, include_endog_lags=True, hmax=HMAX):
    """OLS LP — biased if endog is endogenous, but useful as comparison baseline."""
    common = df_base.index.intersection(endog_s.dropna().index)
    work = df_base[[OUTCOME]+controls].loc[common].copy()
    work['e'] = endog_s.loc[common]
    for l in range(1,4): work[f'Ly{l}'] = work[OUTCOME].shift(l)
    lag_y = [f'Ly{l}' for l in range(1,4)]
    lag_e = []
    if include_endog_lags:
        for l in range(1,3): work[f'Le{l}'] = work['e'].shift(l)
        lag_e = [f'Le{l}' for l in range(1,3)]
    exog_cols = lag_y + lag_e + controls
    rows = []
    for h in range(hmax+1):
        hdf = pd.DataFrame(
            {'y': work[OUTCOME].shift(-h), 'e': work['e'],
             **{c: work[c] for c in exog_cols}}
        ).replace([np.inf,-np.inf],np.nan).dropna()
        if len(hdf) < 40:
            rows.append({'h':h,'coef':np.nan,'se':np.nan}); continue
        X = add_constant(hdf[['e']+exog_cols], has_constant='add')
        fit = sm.OLS(hdf['y'], X).fit(cov_type='HC1')
        rows.append({'h':h,'coef':float(fit.params.get('e',np.nan)),
                     'se':float(fit.bse.get('e',np.nan))})
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef'] - 1.645*irf['se']
    irf['hi90'] = irf['coef'] + 1.645*irf['se']
    return irf


def lp_reduced_form(df_base, instr_s, controls, hmax=HMAX, label=''):
    """Reduced-form LP: WTI on instrument directly. No first stage needed.
    If this is significant, the IV channel exists regardless of instrument strength."""
    common = df_base.index.intersection(instr_s.dropna().index)
    work = df_base[[OUTCOME]+controls].loc[common].copy()
    work['z'] = instr_s.loc[common]
    for l in range(1,4): work[f'Ly{l}'] = work[OUTCOME].shift(l)
    lag_y = [f'Ly{l}' for l in range(1,4)]
    exog_cols = lag_y + controls
    rows = []
    for h in range(hmax+1):
        hdf = pd.DataFrame(
            {'y': work[OUTCOME].shift(-h), 'z': work['z'],
             **{c: work[c] for c in exog_cols}}
        ).replace([np.inf,-np.inf],np.nan).dropna()
        if len(hdf) < 40:
            rows.append({'h':h,'coef':np.nan,'se':np.nan}); continue
        X = add_constant(hdf[['z']+exog_cols], has_constant='add')
        fit = sm.OLS(hdf['y'], X).fit(cov_type='HC1')
        rows.append({'h':h,'coef':float(fit.params.get('z',np.nan)),
                     'se':float(fit.bse.get('z',np.nan))})
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef'] - 1.645*irf['se']
    irf['hi90'] = irf['coef'] + 1.645*irf['se']
    return irf


def ar_ci_grid(y_s, endog_s, instr_s, exog_df, alpha=ALPHA, n_grid=800):
    """
    Anderson-Rubin CI via grid inversion.
    At each beta in the grid, test whether the AR moment condition holds.
    The CI is the set of betas not rejected at level alpha.
    Valid under weak instruments.
    """
    beta_grid = np.linspace(-5, 5, n_grid)
    common = (y_s.dropna().index
              .intersection(endog_s.dropna().index)
              .intersection(instr_s.dropna().index)
              .intersection(exog_df.dropna(how='all').index))
    if len(common) < 40: return np.nan, np.nan
    y = y_s.loc[common].values
    e = endog_s.loc[common].values
    z = instr_s.loc[common].values
    X = add_constant(exog_df.loc[common], has_constant='add').values
    n, k = X.shape
    Xz = np.column_stack([z, X])
    accepted = []
    for b in beta_grid:
        u_b = y - b*e
        try:
            fit = sm.OLS(u_b, Xz).fit(cov_type='HC1')
            Fa = float(fit.f_test('x1 = 0').fvalue)
            pa = 1 - stats.f.cdf(Fa, 1, n-k-1)
            if pa > alpha: accepted.append(b)
        except: pass
    if not accepted: return np.nan, np.nan
    return float(min(accepted)), float(max(accepted))


print('Core functions defined: lp_iv, lp_ols, lp_reduced_form, ar_ci_grid')

Core functions defined: lp_iv, lp_ols, lp_reduced_form, ar_ci_grid


## 7. Dyad-by-dyad LP-IV + OLS + Reduced Form

For each valid dyad we run:
1. **LP-IV** (main causal estimate)
2. **LP-OLS** (biased baseline — IV-OLS difference reveals endogeneity direction)
3. **Reduced-form LP** (instrument → WTI directly — no instrument strength concerns)
4. **Anderson-Rubin CI** at key horizons (weak-instrument-robust)

**Interpretation guide:**
- If IV coef > OLS coef: endogeneity was downward-biasing OLS (measurement error / attenuation)
- If IV coef < OLS coef: endogeneity was upward-biasing OLS (reverse causality / omitted variable)
- If reduced form is significant but IV is not: first stage is too weak to amplify the signal
- If reduced form is insignificant but IV is significant: implausible — likely weak-IV inflation

In [7]:
irf_iv   = {}   # LP-IV
irf_ols  = {}   # LP-OLS
irf_rf   = {}   # reduced form
ar_cis   = {}   # AR-robust CIs at key horizons
summ_rows = []

KEY_H = [0, 6, 12, 24, 36, 48]

print(f'Running LP-IV + OLS + Reduced Form for {len(valid_dyads)} valid dyads...')
print()

for spec in valid_dyads:
    endog = df_raw[spec['lpri_col']].reindex(df_ext.index)
    instr = spec['best_series'].reindex(df_ext.index)

    # LP-IV
    irf_i = lp_iv(df_ext, endog, instr, CTRL_FULL,
                  include_endog_lags=spec['can_elags'], label=spec['name'])
    irf_iv[spec['code']] = irf_i

    # LP-OLS
    irf_o = lp_ols(df_ext, endog, CTRL_FULL,
                   include_endog_lags=spec['can_elags'])
    irf_ols[spec['code']] = irf_o

    # Reduced form
    irf_r = lp_reduced_form(df_ext, instr, CTRL_FULL, label=spec['name'])
    irf_rf[spec['code']] = irf_r

    # AR-robust CIs
    lag_y = [f'Ly{l}' for l in range(1,4)]
    lag_e = [f'Le{l}' for l in range(1,3)] if spec['can_elags'] else []
    work = df_ext[[OUTCOME]+CTRL_FULL].copy()
    work['e'] = endog; work['z'] = instr
    for l in range(1,4): work[f'Ly{l}'] = work[OUTCOME].shift(l)
    if spec['can_elags']:
        for l in range(1,3): work[f'Le{l}'] = work['e'].shift(l)

    ar_rows = []
    for h in KEY_H:
        work['y_fwd'] = work[OUTCOME].shift(-h)
        sub = work[['y_fwd','e','z']+lag_y+lag_e+CTRL_FULL].dropna()
        ar_lo, ar_hi = ar_ci_grid(sub['y_fwd'], sub['e'], sub['z'],
                                   sub[lag_y+lag_e+CTRL_FULL])
        wald_lo = float(irf_i.loc[irf_i['h']==h,'lo90'].values[0]) if len(irf_i) > h else np.nan
        wald_hi = float(irf_i.loc[irf_i['h']==h,'hi90'].values[0]) if len(irf_i) > h else np.nan
        ar_rows.append({'h':h,'wald_lo':wald_lo,'wald_hi':wald_hi,
                        'ar_lo':ar_lo,'ar_hi':ar_hi})
    ar_cis[spec['code']] = pd.DataFrame(ar_rows)
    ar_cis[spec['code']].to_csv(RESULTS/f'ar_ci_{spec["code"]}.csv', index=False)

    # Summary stats
    sig_iv  = int((irf_i['lo90']>0).sum()+(irf_i['hi90']<0).sum())
    sig_ols = int((irf_o['lo90']>0).sum()+(irf_o['hi90']<0).sum())
    sig_rf  = int((irf_r['lo90']>0).sum()+(irf_r['hi90']<0).sum())
    F_min   = round(float(irf_i['F'].min()),1)

    # OLS vs IV bias: positive = OLS understates, negative = OLS overstates
    h12_iv  = float(irf_i.loc[irf_i['h']==12,'coef'].values[0])
    h12_ols = float(irf_o.loc[irf_o['h']==12,'coef'].values[0])
    bias_dir = 'IV>OLS (attenuation)' if h12_iv > h12_ols else 'IV<OLS (rev.causal.)'

    irf_i.to_csv(RESULTS/f'irf_iv_{spec["code"]}.csv',  index=False)
    irf_o.to_csv(RESULTS/f'irf_ols_{spec["code"]}.csv', index=False)
    irf_r.to_csv(RESULTS/f'irf_rf_{spec["code"]}.csv',  index=False)

    warn = '  ⚠F_min<10' if F_min < MIN_F else ''
    print(f'  {spec["name"]:<22}  IV sig90={sig_iv:2d}  OLS sig90={sig_ols:2d}'
          f'  RF sig90={sig_rf:2d}  F_min={F_min}{warn}')
    print(f'    h=12: IV={h12_iv:.4f}  OLS={h12_ols:.4f}  → {bias_dir}')

    summ_rows.append(dict(
        name=spec['name'], code=spec['code'], instrument=spec['best_name'],
        F_diag=round(spec['best_F'],1), F_min_lp=F_min,
        sig_iv=sig_iv, sig_ols=sig_ols, sig_rf=sig_rf,
        h12_iv=round(h12_iv,4), h12_ols=round(h12_ols,4),
        bias_dir=bias_dir,
        strong=spec['strong']
    ))

summ_df = pd.DataFrame(summ_rows)
summ_df.to_csv(RESULTS/'panel_dyad_summary.csv', index=False)
print()
print(summ_df[['name','instrument','F_diag','F_min_lp','sig_iv','sig_ols','sig_rf']].to_string(index=False))

Running LP-IV + OLS + Reduced Form for 6 valid dyads...

  US–China                IV sig90=16  OLS sig90=20  RF sig90= 0  F_min=242.7
    h=12: IV=-0.0229  OLS=-0.0059  → IV<OLS (rev.causal.)
  Japan–China             IV sig90= 0  OLS sig90= 0  RF sig90= 0  F_min=101.5
    h=12: IV=-0.2118  OLS=-0.2248  → IV>OLS (attenuation)
  Australia–China         IV sig90= 2  OLS sig90=35  RF sig90= 1  F_min=5.7  ⚠F_min<10
    h=12: IV=-0.1649  OLS=0.1888  → IV<OLS (rev.causal.)
  France–China            IV sig90= 0  OLS sig90=45  RF sig90= 0  F_min=10.4
    h=12: IV=-0.1168  OLS=0.4971  → IV<OLS (rev.causal.)
  Germany–China           IV sig90= 3  OLS sig90=46  RF sig90= 3  F_min=12.5
    h=12: IV=1.7449  OLS=2.5087  → IV<OLS (rev.causal.)
  Russia–China            IV sig90=26  OLS sig90=41  RF sig90=26  F_min=5.8  ⚠F_min<10
    h=12: IV=0.5978  OLS=0.2395  → IV>OLS (attenuation)

           name instrument  F_diag  F_min_lp  sig_iv  sig_ols  sig_rf
       US–China      d2pri   242.7     242.7  

### Figure: LP-IV vs OLS vs Reduced Form per dyad

Three overlaid IRFs per dyad. The gap between IV and OLS indicates endogeneity bias.
The reduced form shows the instrument-WTI channel before scaling by the first stage.

In [8]:
hs  = np.arange(HMAX+1)
n_v = len(valid_dyads)
if n_v == 0:
    print('No valid dyads.')
else:
    ncols = min(3, n_v)
    nrows = int(np.ceil(n_v / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(8*ncols, 6*nrows), squeeze=False)
    DCOLS = ['steelblue','firebrick','darkorange','teal','purple','darkgreen','brown','olive']

    for i, spec in enumerate(valid_dyads):
        ax   = axes[i//ncols][i%ncols]
        iv   = irf_iv[spec['code']]
        ols  = irf_ols[spec['code']]
        rf   = irf_rf[spec['code']]
        col  = DCOLS[i%len(DCOLS)]
        sig90 = int((iv['lo90']>0).sum()+(iv['hi90']<0).sum())
        sig_mask = (iv['lo90']>0)|(iv['hi90']<0)

        ax.plot(hs, iv['coef'],  color=col,         lw=2,   label=f'LP-IV (sig={sig90})')
        ax.fill_between(hs, iv['lo90'], iv['hi90'], color=col, alpha=0.18)
        ax.plot(hs, ols['coef'], color='grey',       lw=1.5, linestyle='--', label='LP-OLS')
        ax.plot(hs, rf['coef'],  color='black',      lw=1,   linestyle=':',  label='Reduced form')
        ax.scatter(hs[sig_mask.values], iv['coef'][sig_mask.values],
                   color=col, edgecolors='red', zorder=5, s=20)

        # AR CI markers
        if spec['code'] in ar_cis:
            for _, row in ar_cis[spec['code']].iterrows():
                if not pd.isna(row['ar_lo']):
                    ax.plot([row['h'],row['h']], [row['ar_lo'],row['ar_hi']],
                            color='crimson', lw=3, alpha=0.5)
            ax.plot([],[], color='crimson', lw=3, alpha=0.5, label='AR-robust CI')

        fmin = summ_df.loc[summ_df['code']==spec['code'],'F_min_lp'].values[0]
        ax.set_facecolor('#fff8f8' if fmin < MIN_F else 'white')
        ax.axhline(0, color='black', lw=0.8)
        ax.set_xlim(0, HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
        ax.set_xlabel('Horizon (months)', fontsize=9)
        ax.set_ylabel('Coefficient', fontsize=9)
        ax.set_title(f'{spec["name"]} | {spec["best_name"]} F={spec["best_F"]:.0f}'
                     f'\\nIV sig90={int((iv["lo90"]>0).sum()+(iv["hi90"]<0).sum())}  '
                     f'OLS={int((ols["lo90"]>0).sum()+(ols["hi90"]<0).sum())}  '
                     f'RF={int((rf["lo90"]>0).sum()+(rf["hi90"]<0).sum())}', fontsize=9)
        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    for j in range(n_v, nrows*ncols):
        axes[j//ncols][j%ncols].set_visible(False)

    plt.suptitle('Dyad LP-IV vs OLS vs Reduced Form (90% CI + AR-robust bounds)\\n'
                 'Light red = F drops below 10 at some horizons', fontsize=10, y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES/'Figure_10_dyad_full.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: Figure_10_dyad_full.png')

Saved: Figure_10_dyad_full.png


## 8. Specification curve — US-China

We run the US-China LP-IV under every control specification in `CTRL_SETS`.
This answers: does the US-China finding depend on which controls we include?

A robust finding should show consistent sig90 and consistent h=12 coefficient
across all specifications. Sensitivity to control choice would suggest the
result is driven by control selection rather than the instrument.

In [9]:
us_spec  = next((r for r in valid_dyads if r['code']=='us'), None)
if us_spec is None:
    print('US-China not valid — specification curve skipped.')
    spec_curve = {}
else:
    endog_us = df_raw[us_spec['lpri_col']].reindex(df_ext.index)
    instr_us = us_spec['best_series'].reindex(df_ext.index)

    # Add GPR-only and WUI-only as special cases if available
    extra_specs = {}
    if 'gpr_chn_l1' in df_ext.columns:
        extra_specs['GPR-China only (1)'] = ['gpr_chn_l1']
    if 'gpr_usa_l1' in df_ext.columns:
        extra_specs['GPR-USA only (1)'] = ['gpr_usa_l1']
    all_specs = {**CTRL_SETS, **extra_specs}

    spec_curve = {}
    print('SPECIFICATION CURVE — US-China LP-IV')
    print('='*75)
    print(f'  {"Control set":<22} {"n_ctrl":>7} {"sig90":>7} {"β(h=6)":>10} {"β(h=12)":>10} {"F_min":>8}')
    print('-'*75)

    for sname, sctrl in all_specs.items():
        # Only keep controls that exist in df_ext
        sctrl_ok = [c for c in sctrl if c in df_ext.columns]
        if not sctrl_ok: continue
        df_s = df_ext_nlp if 'NLP' in sname else df_ext
        irf_s = lp_iv(df_s, endog_us, instr_us, sctrl_ok,
                      include_endog_lags=us_spec['can_elags'], label=sname)
        spec_curve[sname] = irf_s
        irf_s.to_csv(RESULTS/f'irf_us_spec_{sname[:15].replace(" ","_")}.csv', index=False)

        sig  = int((irf_s['lo90']>0).sum()+(irf_s['hi90']<0).sum())
        Fm   = round(float(irf_s['F'].min()),1)
        h6   = float(irf_s.loc[irf_s['h']==6,'coef'].values[0])
        h12  = float(irf_s.loc[irf_s['h']==12,'coef'].values[0])
        print(f'  {sname:<22} {len(sctrl_ok):>7} {sig:>7} {h6:>10.4f} {h12:>10.4f} {Fm:>8}')

    # Spec curve figure
    hs = np.arange(HMAX+1)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    SCOLS = plt.cm.tab10(np.linspace(0, 1, len(spec_curve)))
    for (sname, irf_s), col in zip(spec_curve.items(), SCOLS):
        sig = int((irf_s['lo90']>0).sum()+(irf_s['hi90']<0).sum())
        axes[0].plot(hs, irf_s['coef'], color=col, lw=1.5, label=f'{sname} ({sig}/49)')
    axes[0].axhline(0, color='black', lw=0.8)
    axes[0].set_xlim(0,HMAX); axes[0].set_xticks(np.arange(0,HMAX+1,12))
    axes[0].set_title('US-China LP-IV IRF — all specifications')
    axes[0].legend(fontsize=7); axes[0].grid(alpha=0.2)

    # sig90 by spec
    snames = list(spec_curve.keys())
    sig90s = [int((v['lo90']>0).sum()+(v['hi90']<0).sum()) for v in spec_curve.values()]
    axes[1].barh(range(len(snames)), sig90s, color=SCOLS[:len(snames)])
    axes[1].set_yticks(range(len(snames))); axes[1].set_yticklabels(snames, fontsize=9)
    axes[1].axvline(4.9, color='black', lw=1, linestyle='--', label='~5% false positive')
    axes[1].set_xlabel('sig90 (out of 49 horizons)'); axes[1].set_title('Significant horizons by specification')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.2)

    plt.suptitle('Specification Curve: US–China LP-IV\\nConsistency across control sets = robust finding', fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES/'Figure_10_spec_curve.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: Figure_10_spec_curve.png')

SPECIFICATION CURVE — US-China LP-IV
  Control set             n_ctrl   sig90     β(h=6)    β(h=12)    F_min
---------------------------------------------------------------------------
  minimal (3)                  3      18    -0.1240    -0.0145    243.4
  core+fin (7)                 7      15    -0.1374    -0.0239    238.7
  core+macro (12)             12      17    -0.1624    -0.0233    240.2
  full (14)                   14      16    -0.1631    -0.0229    242.7
  full+NLP (18)               14      16    -0.1631    -0.0229    242.7
  GPR-China only (1)           1      13    -0.1678    -0.0777    251.6
  GPR-USA only (1)             1      13    -0.1824    -0.1299    250.3
Saved: Figure_10_spec_curve.png


## 9. Placebo test — permuted instrument

We randomly permute the US-China `d2pri` series (break the time structure) and
re-run the LP-IV 200 times. We then ask: how often does the permuted instrument
produce sig90 ≥ the observed 16?

If the answer is < 5%, the US-China result is unlikely to be a chance finding.
If it is > 5%, the instrument may be picking up spurious correlations.

This is the most direct test of whether the result is real.

In [10]:
if us_spec is None:
    print('Placebo skipped: US-China not valid.')
else:
    N_PERM = 200
    endog_us = df_raw[us_spec['lpri_col']].reindex(df_ext.index)
    instr_us = us_spec['best_series'].reindex(df_ext.index)

    # Observed sig90
    irf_obs = irf_iv['us']
    sig_obs = int((irf_obs['lo90']>0).sum()+(irf_obs['hi90']<0).sum())

    print(f'PLACEBO TEST: {N_PERM} permutations of US-China d2pri')
    print(f'Observed sig90 = {sig_obs}/49')
    print('Running permutations...')

    perm_sig90s = []
    instr_vals  = instr_us.dropna().values.copy()
    instr_idx   = instr_us.dropna().index

    for perm_i in range(N_PERM):
        if perm_i % 50 == 0: print(f'  {perm_i}/{N_PERM}...', end=' ', flush=True)
        perm_vals   = np.random.permutation(instr_vals)
        instr_perm  = pd.Series(perm_vals, index=instr_idx).reindex(df_ext.index)
        try:
            irf_p = lp_iv(df_ext, endog_us, instr_perm, CTRL_FULL,
                           include_endog_lags=us_spec['can_elags'])
            perm_sig90s.append(int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum()))
        except:
            perm_sig90s.append(0)

    print('done.')
    perm_arr = np.array(perm_sig90s)
    p_val    = (perm_arr >= sig_obs).mean()
    rank_pct = (perm_arr < sig_obs).mean() * 100

    print()
    print(f'PERMUTATION TEST RESULTS')
    print(f'  Observed sig90           : {sig_obs}/49')
    print(f'  Permutation sig90 mean   : {perm_arr.mean():.1f}')
    print(f'  Permutation sig90 median : {np.median(perm_arr):.0f}')
    print(f'  Permutation sig90 95th % : {np.percentile(perm_arr,95):.0f}')
    print(f'  p-value (share ≥ {sig_obs}) : {p_val:.3f}')
    print(f'  Rank percentile          : {rank_pct:.1f}th percentile')
    print()
    if p_val < 0.05:
        print(f'RESULT: Observed sig90={sig_obs} exceeds {rank_pct:.0f}% of permutations.')
        print('  The US-China result is unlikely to be a chance finding (p={:.3f}).'.format(p_val))
    else:
        print(f'RESULT: p={p_val:.3f} — cannot rule out a chance finding.')

    pd.DataFrame({'perm_sig90': perm_arr}).to_csv(RESULTS/'placebo_perm_sig90.csv', index=False)

    fig, ax = plt.subplots(figsize=(9,4))
    ax.hist(perm_arr, bins=range(0,30), color='steelblue', alpha=0.7, edgecolor='white',
            label=f'Permuted sig90 (n={N_PERM})')
    ax.axvline(sig_obs, color='firebrick', lw=2, label=f'Observed sig90={sig_obs}')
    ax.axvline(np.percentile(perm_arr,95), color='orange', lw=1.5, linestyle='--',
               label=f'95th percentile={np.percentile(perm_arr,95):.0f}')
    ax.set_xlabel('sig90 (horizons significant at 90% CI)')
    ax.set_ylabel('Count')
    ax.set_title(f'Placebo Test — US-China LP-IV\\np={p_val:.3f}, rank={rank_pct:.0f}th percentile')
    ax.legend(fontsize=9); ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIGURES/'Figure_10_placebo.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: Figure_10_placebo.png')

PLACEBO TEST: 200 permutations of US-China d2pri
Observed sig90 = 16/49
Running permutations...
  0/200...   50/200...   100/200...   150/200... done.

PERMUTATION TEST RESULTS
  Observed sig90           : 16/49
  Permutation sig90 mean   : 0.0
  Permutation sig90 median : 0
  Permutation sig90 95th % : 0
  p-value (share ≥ 16) : 0.000
  Rank percentile          : 100.0th percentile

RESULT: Observed sig90=16 exceeds 100% of permutations.
  The US-China result is unlikely to be a chance finding (p=0.000).
Saved: Figure_10_placebo.png


## 10. External validity: Wald test H2 (US-China vs Japan-China)

**H2:** $\beta_{US}(h) = \beta_{JP}(h)$ for all $h$.

If Japan-China shows 0/49 significant horizons despite a strong instrument (F=112),
this is itself a finding: the geopolitical transmission mechanism is US-specific,
not a universal bilateral phenomenon. We test this formally and descriptively.

In [11]:
if 'us' in irf_iv and 'jp' in irf_iv:
    wald_rows = []
    for h in range(HMAX+1):
        cu,su = float(irf_iv['us'].loc[h,'coef']), float(irf_iv['us'].loc[h,'se'])
        cj,sj = float(irf_iv['jp'].loc[h,'coef']), float(irf_iv['jp'].loc[h,'se'])
        if any(pd.isna([cu,su,cj,sj])) or su==0 or sj==0:
            wald_rows.append({'h':h,'diff':np.nan,'z':np.nan,'p':np.nan,'sig10':False}); continue
        diff=cu-cj; se=np.sqrt(su**2+sj**2)
        z=diff/se; p=float(2*(1-stats.norm.cdf(abs(z))))
        wald_rows.append({'h':h,'diff':diff,'z':z,'p':p,'sig10':p<ALPHA})
    wald_h2 = pd.DataFrame(wald_rows)
    wald_h2.to_csv(RESULTS/'wald_us_vs_jp.csv', index=False)
    sig_h2 = int(wald_h2['sig10'].sum())

    jp_sig = int((irf_iv['jp']['lo90']>0).sum()+(irf_iv['jp']['hi90']<0).sum())
    us_sig = int((irf_iv['us']['lo90']>0).sum()+(irf_iv['us']['hi90']<0).sum())

    print('H2 EXTERNAL VALIDITY — US-China vs Japan-China')
    print('='*65)
    print(f'US-China  : sig90={us_sig}/49  F={next(r["best_F"] for r in valid_dyads if r["code"]=="us"):.0f}')
    print(f'Japan-China: sig90={jp_sig}/49  F={next(r["best_F"] for r in valid_dyads if r["code"]=="jp"):.0f}')
    print(f'Wald H2 rejected at 10%: {sig_h2}/49 horizons (expected ~5 by chance)')
    print()
    if jp_sig == 0:
        print('FINDING: Japan-China shows ZERO significant horizons despite F=112 instrument.')
        print('  This is a genuine null — not a power issue (F=112 is very strong).')
        print('  Interpretation: geopolitical turning points in Japan-China relations')
        print('  do not transmit to WTI prices the way US-China turning points do.')
        print('  The US-China channel may be specific to the dominant oil-importing')
        print('  bilateral relationship, not a general mechanism.')
    elif sig_h2 <= 5:
        print('FINDING: Cannot reject H2. Japan-China validates US-China.')
    else:
        print(f'FINDING: H2 rejected at {sig_h2}/49 horizons — dyad-specific heterogeneity.')

    print()
    print('Selected horizons:')
    print(wald_h2[wald_h2['h'].isin([0,6,12,18,24,36,48])][
          ['h','diff','z','p','sig10']].to_string(index=False))
else:
    sig_h2 = None
    print('H2 skipped: missing US or JP in valid dyads.')

H2 EXTERNAL VALIDITY — US-China vs Japan-China
US-China  : sig90=16/49  F=243
Japan-China: sig90=0/49  F=113
Wald H2 rejected at 10%: 1/49 horizons (expected ~5 by chance)

FINDING: Japan-China shows ZERO significant horizons despite F=112 instrument.
  This is a genuine null — not a power issue (F=112 is very strong).
  Interpretation: geopolitical turning points in Japan-China relations
  do not transmit to WTI prices the way US-China turning points do.
  The US-China channel may be specific to the dominant oil-importing
  bilateral relationship, not a general mechanism.

Selected horizons:
 h      diff         z        p  sig10
 0 -0.057969 -1.028870 0.303541  False
 6 -0.005771 -0.039045 0.968855  False
12  0.188864  1.084636 0.278083  False
18  0.224347  1.054158 0.291810  False
24  0.281797  1.200908 0.229787  False
36  0.093475  0.370257 0.711191  False
48 -0.272291 -1.268207 0.204724  False


## 11. Meta-analytic pooling (IVW + Cochran Q)

Before imposing a common slope via the CF panel, we test whether dyad effects
are homogeneous using Cochran Q. High I² means pooling imposes a false restriction.

We run IVW separately for:
- All 6 valid dyads
- Strong dyads only (US + Japan, F≥30)
- Valid minus Russia/Australia (those with F_min_lp < 10 at long horizons)

In [12]:
def ivw_pool(irf_dict, codes, hmax=HMAX, label=''):
    rows = []
    for h in range(hmax+1):
        betas, ses = [], []
        for c in codes:
            if c not in irf_dict: continue
            b = float(irf_dict[c].loc[h,'coef']) if h < len(irf_dict[c]) else np.nan
            s = float(irf_dict[c].loc[h,'se'])   if h < len(irf_dict[c]) else np.nan
            if not any(pd.isna([b,s])) and s > 0:
                betas.append(b); ses.append(s)
        if len(betas) < 2:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'Q':np.nan,'Q_p':np.nan,'I2':np.nan,'k':len(betas)}); continue
        w = np.array([1/s**2 for s in ses]); b_arr = np.array(betas)
        b_ivw = np.sum(w*b_arr)/np.sum(w); se_ivw = np.sqrt(1/np.sum(w))
        Q = np.sum(w*(b_arr-b_ivw)**2); k = len(betas)
        Q_p = 1-stats.chi2.cdf(Q, df=k-1); I2 = max(0,(Q-(k-1))/Q)*100
        rows.append({'h':h,'coef':b_ivw,'se':se_ivw,'Q':Q,'Q_p':Q_p,'I2':I2,'k':k})
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef']-1.645*irf['se']
    irf['hi90'] = irf['coef']+1.645*irf['se']
    return irf

# Three pooling sets
pool_sets = {
    'All valid (6 dyads)':      [r['code'] for r in valid_dyads],
    'Strong only (US+JP)':      [r['code'] for r in strong_dyads],
    'Valid, stable F (US+JP+GER+FRA)': [r['code'] for r in valid_dyads
                                         if r['code'] not in ('aus','rus')],
}

irf_ivw = {}
print('META-ANALYTIC POOLING (Inverse-Variance Weighted)')
print('='*70)
print(f'  {"Pool":<35} {"sig90":>7} {"Q_sig(10%)":>11} {"Mean I²":>9}')
print('-'*70)

for pname, pcodes in pool_sets.items():
    pcodes = [c for c in pcodes if c in irf_iv]
    if len(pcodes) < 2:
        print(f'  {pname:<35}  SKIPPED (fewer than 2 valid dyads)')
        continue
    irf_p = ivw_pool(irf_iv, pcodes, label=pname)
    irf_ivw[pname] = irf_p
    irf_p.to_csv(RESULTS/f'irf_ivw_{pname[:20].replace(" ","_")}.csv', index=False)
    sig   = int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum())
    Qsig  = int((irf_p['Q_p']<ALPHA).sum())
    mI2   = irf_p['I2'].mean()
    print(f'  {pname:<35}  {sig:>7}  {Qsig:>11}  {mI2:>9.1f}%')

print()
print('Interpretation:')
print('  I² > 75%: high heterogeneity — common-slope panel is unjustified')
print('  I² 25-75%: moderate — panel with caution')
print('  I² < 25%: low — pooling defensible')

META-ANALYTIC POOLING (Inverse-Variance Weighted)
  Pool                                  sig90  Q_sig(10%)   Mean I²
----------------------------------------------------------------------
  All valid (6 dyads)                       18           15       20.3%
  Strong only (US+JP)                       15            0        2.9%
  Valid, stable F (US+JP+GER+FRA)           14            0        6.4%

Interpretation:
  I² > 75%: high heterogeneity — common-slope panel is unjustified
  I² 25-75%: moderate — panel with caution
  I² < 25%: low — pooling defensible


### Figure: IVW IRFs for all three pooling sets + heterogeneity (I²)

In [13]:
hs = np.arange(HMAX+1)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
pool_colors = {'All valid (6 dyads)': 'steelblue',
               'Strong only (US+JP)': 'firebrick',
               'Valid, stable F (US+JP+GER+FRA)': 'darkorange'}
for pname, irf_p in irf_ivw.items():
    sig = int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum())
    col = pool_colors.get(pname, 'grey')
    ax.plot(hs, irf_p['coef'], color=col, lw=2, label=f'{pname} (sig90={sig})')
    ax.fill_between(hs, irf_p['lo90'], irf_p['hi90'], color=col, alpha=0.10)
if 'us' in irf_iv:
    ax.plot(hs, irf_iv['us']['coef'], color='grey', lw=1, linestyle=':', alpha=0.5,
            label='US-China only (ref)')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlim(0,HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
ax.set_xlabel('Horizon (months)'); ax.set_ylabel('IVW coefficient')
ax.set_title('Meta-analytic IVW: Three pooling sets\\n(90% CI)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

ax2 = axes[1]
ref_pool = list(irf_ivw.keys())[0] if irf_ivw else None
if ref_pool:
    irf_ref = irf_ivw[ref_pool]
    ax2.bar(hs, irf_ref['I2'].fillna(0),
            color=['firebrick' if p < ALPHA else 'steelblue' for p in irf_ref['Q_p'].fillna(1)],
            alpha=0.7, label='I² by horizon')
    ax2.axhline(75, color='firebrick', lw=1.5, linestyle='--', label='I²=75 (high heterogeneity)')
    ax2.axhline(50, color='orange',    lw=1.5, linestyle='--', label='I²=50 (moderate)')
    ax2.axhline(25, color='steelblue', lw=1.5, linestyle='--', label='I²=25 (low)')
    ax2.set_title(f'Cross-dyad heterogeneity (Cochran I²)\\n{ref_pool}\\n'
                  f'Red = Q significant at 10%')
    ax2.legend(fontsize=8)
ax2.set_xlim(0,HMAX); ax2.set_xticks(np.arange(0,HMAX+1,12))
ax2.set_xlabel('Horizon (months)'); ax2.set_ylabel('I² (%)')
ax2.grid(alpha=0.2)

plt.suptitle('IVW Meta-Analytic Pooling — All Configurations', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_10_ivw.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_10_ivw.png')

Saved: Figure_10_ivw.png


## 12. CF pooled panel — strong dyads only vs all valid

We run the control-function panel twice:
1. **Strong dyads only** (US + Japan) — homogeneous, both with d2pri instrument
2. **All valid dyads** — includes borderline instruments, heterogeneous

The comparison tells us how much the weak dyads contaminate the pooled estimate.

In [14]:
def run_cf_panel(spec_list, df_base, label='CF panel', hmax=HMAX):
    rows = []
    for h in range(hmax+1):
        if h % 12 == 0: print(f'  h={h}...', end=' ', flush=True)
        frames = []
        for spec in spec_list:
            df_d = df_base[[OUTCOME]+CTRL_FULL].copy()
            df_d['e']    = df_raw[spec['lpri_col']].reindex(df_base.index)
            df_d['z']    = spec['best_series'].reindex(df_base.index)
            df_d['dyad'] = spec['code']
            for l in range(1,4): df_d[f'Ly{l}'] = df_d[OUTCOME].shift(l)
            df_d['y_fwd'] = df_d[OUTCOME].shift(-h)
            frames.append(df_d)
        panel = pd.concat(frames).replace([np.inf,-np.inf],np.nan)
        lag_y  = [f'Ly{l}' for l in range(1,4)]
        exog_p = lag_y + CTRL_FULL
        panel  = panel[['y_fwd','e','z','dyad']+exog_p].dropna()
        if len(panel) < 60:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)}); continue
        try:
            panel = panel.copy(); panel['cf_r'] = np.nan
            for d in panel['dyad'].unique():
                mask = panel['dyad']==d
                X_fs = add_constant(panel[mask][['z']+exog_p], has_constant='add')
                panel.loc[mask,'cf_r'] = sm.OLS(panel[mask]['e'], X_fs).fit().resid
            panel = panel.dropna(subset=['cf_r'])
            dums  = pd.get_dummies(panel['dyad'], prefix='D', drop_first=True).astype(float)
            panel = pd.concat([panel.reset_index(drop=True), dums.reset_index(drop=True)], axis=1)
            X2 = add_constant(panel[['e','cf_r']+exog_p+dums.columns.tolist()], has_constant='add')
            fit2 = sm.OLS(panel['y_fwd'], X2).fit(cov_type='HC1')
            rows.append({'h':h,'coef':float(fit2.params.get('e',np.nan)),
                         'se':float(fit2.bse.get('e',np.nan)),'n':len(panel)})
        except Exception as ex:
            print(f'\n  [{label} h={h}] {type(ex).__name__}: {ex}')
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)})
    print(' done.')
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef']-1.645*irf['se']
    irf['hi90'] = irf['coef']+1.645*irf['se']
    return irf

cf_panels = {}

# Strong dyads only (US + Japan)
if len(strong_dyads) >= 2:
    print(f'CF panel — strong dyads ({len(strong_dyads)}):')
    cf_panels['Strong (US+JP)'] = run_cf_panel(strong_dyads, df_ext, 'CF strong')
    cf_panels['Strong (US+JP)'].to_csv(RESULTS/'irf_cf_strong.csv', index=False)
else:
    print(f'Only {len(strong_dyads)} strong dyads — strong-only CF requires ≥ 2.')

# All valid dyads
print(f'CF panel — all valid ({len(valid_dyads)} dyads):')
cf_panels['All valid (6)'] = run_cf_panel(valid_dyads, df_ext, 'CF all')
cf_panels['All valid (6)'].to_csv(RESULTS/'irf_cf_all.csv', index=False)

print()
for pname, irf_p in cf_panels.items():
    sig = int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum())
    nan = int(irf_p['coef'].isna().sum())
    n   = irf_p['n'].mean()
    print(f'  {pname:<30}  sig90={sig}/49  NaN={nan}  n_mean={n:.0f}')

CF panel — strong dyads (3):
  h=0...   h=12...   h=24...   h=36...   h=48...  done.
CF panel — all valid (6 dyads):
  h=0...   h=12...   h=24...   h=36...   h=48...  done.

  Strong (US+JP)                  sig90=0/49  NaN=0  n_mean=1074
  All valid (6)                   sig90=0/49  NaN=0  n_mean=2148


## 13. Panel DML-PLIV — XGBoost + Ridge, stability diagnostics

We run DML-PLIV with two learner types:
1. **XGBoost** (flexible, nonlinear)
2. **Ridge regression** (stable, linear — serves as a sanity check)

If Ridge and XGBoost produce similar estimates, the linear CF panel is adequate.
If XGBoost differs substantially, nonlinearity in the nuisance functions matters.

We also report the DML SE for each horizon — large SEs indicate instability,
usually caused by weak instruments inflating the PLIV step.

In [15]:
def get_xgb_stable():
    return XGBRegressor(n_estimators=150, max_depth=2, learning_rate=0.05,
                        subsample=0.7, colsample_bytree=0.7,
                        verbosity=0, random_state=42, n_jobs=-1)

def get_ridge():
    from sklearn.linear_model import Ridge
    return Ridge(alpha=1.0)

def run_dml_panel(spec_list, df_base, ml_fn, label='DML', hmax=HMAX):
    # Build base panel
    frames = []
    for spec in spec_list:
        df_d = df_base[[OUTCOME]+CTRL_FULL].copy()
        df_d['lpri_p']  = df_raw[spec['lpri_col']].reindex(df_base.index)
        df_d['instr_p'] = spec['best_series'].reindex(df_base.index)
        df_d['dyad']    = spec['code']
        for l in range(1,4): df_d[f'Ly{l}'] = df_d[OUTCOME].shift(l)
        frames.append(df_d)
    base = pd.concat(frames)
    lag_y = [f'Ly{l}' for l in range(1,4)]
    dums  = pd.get_dummies(base['dyad'], prefix='D', drop_first=True).astype(float)
    base  = pd.concat([base.reset_index(drop=True), dums.reset_index(drop=True)], axis=1)
    X_DML = lag_y + CTRL_FULL + dums.columns.tolist()

    rows = []
    print(f'{label} ({len(spec_list)} dyads, h=0–{hmax})...')
    for h in range(hmax+1):
        if h % 12 == 0: print(f'  h={h}...', end=' ', flush=True)
        base['y_fwd'] = base.groupby('dyad')[OUTCOME].transform(lambda s: s.shift(-h))
        sub = base[['y_fwd','lpri_p','instr_p']+X_DML].replace([np.inf,-np.inf],np.nan).dropna()
        if len(sub) < 80:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)}); continue
        try:
            data_obj = dml.DoubleMLData(sub, y_col='y_fwd', d_cols='lpri_p',
                                         z_cols='instr_p', x_cols=X_DML)
            pliv = dml.DoubleMLPLIV(data_obj,
                                     ml_l=ml_fn(), ml_m=ml_fn(), ml_r=ml_fn(),
                                     n_folds=3, n_rep=3)
            pliv.fit()
            rows.append({'h':h,'coef':float(pliv.coef[0]),'se':float(pliv.se[0]),'n':len(sub)})
        except Exception as ex:
            print(f'\n  [{label} h={h}] {type(ex).__name__}: {ex}')
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
    print(' done.')
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef']-1.645*irf['se']
    irf['hi90'] = irf['coef']+1.645*irf['se']
    return irf

dml_results = {}

# Run on all valid dyads
dml_results['XGBoost (all valid)'] = run_dml_panel(valid_dyads, df_ext, get_xgb_stable,
                                                     label='DML-PLIV XGBoost')
dml_results['XGBoost (all valid)'].to_csv(RESULTS/'irf_dml_xgb.csv', index=False)

dml_results['Ridge (all valid)'] = run_dml_panel(valid_dyads, df_ext, get_ridge,
                                                   label='DML-PLIV Ridge')
dml_results['Ridge (all valid)'].to_csv(RESULTS/'irf_dml_ridge.csv', index=False)

# Strong dyads only
if len(strong_dyads) >= 2:
    dml_results['XGBoost (strong)'] = run_dml_panel(strong_dyads, df_ext, get_xgb_stable,
                                                      label='DML-PLIV XGBoost (strong)')
    dml_results['XGBoost (strong)'].to_csv(RESULTS/'irf_dml_xgb_strong.csv', index=False)

print()
print('DML RESULTS SUMMARY')
print(f'  {"Model":<30} {"sig90":>7} {"NaN":>5} {"median SE":>12} {"max |coef|":>12}')
for mname, irf_m in dml_results.items():
    sig  = int((irf_m['lo90']>0).sum()+(irf_m['hi90']<0).sum())
    nan  = int(irf_m['coef'].isna().sum())
    mse  = irf_m['se'].median()
    mc   = irf_m['coef'].abs().max()
    flag = '  ⚠ unstable' if mse > 1.0 else ''
    print(f'  {mname:<30} {sig:>7} {nan:>5} {mse:>12.4f} {mc:>12.4f}{flag}')

# Wald: CF vs DML (reference models)
cf_ref  = cf_panels.get('All valid (6)', pd.DataFrame())
dml_ref = dml_results.get('XGBoost (all valid)', pd.DataFrame())
if not cf_ref.empty and not dml_ref.empty:
    wald_cd = []
    for h in range(HMAX+1):
        cc,sc = float(cf_ref.loc[h,'coef']),  float(cf_ref.loc[h,'se'])
        cd,sd = float(dml_ref.loc[h,'coef']), float(dml_ref.loc[h,'se'])
        if any(pd.isna([cc,sc,cd,sd])) or sc==0 or sd==0:
            wald_cd.append({'h':h,'p':np.nan,'sig10':False}); continue
        z=( cc-cd)/np.sqrt(sc**2+sd**2); p=float(2*(1-stats.norm.cdf(abs(z))))
        wald_cd.append({'h':h,'diff':cc-cd,'p':p,'sig10':p<ALPHA})
    w_cd = pd.DataFrame(wald_cd)
    sig_wd = int(w_cd['sig10'].sum())
    w_cd.to_csv(RESULTS/'wald_cf_vs_dml.csv', index=False)
    print(f'\nWald CF≠DML: {sig_wd}/49 at 10%  →  {"nonlinearity present" if sig_wd>5 else "linear CF adequate"}')
else:
    sig_wd = None

DML-PLIV XGBoost (6 dyads, h=0–48)...
  h=0...   h=12...   h=24...   h=36...   h=48...  done.
DML-PLIV Ridge (6 dyads, h=0–48)...
  h=0...   h=12...   h=24...   h=36...   h=48...  done.
DML-PLIV XGBoost (strong) (3 dyads, h=0–48)...
  h=0...   h=12...   h=24...   h=36...   h=48...  done.

DML RESULTS SUMMARY
  Model                            sig90   NaN    median SE   max |coef|
  XGBoost (all valid)                  0     0       0.9136       0.5560
  Ridge (all valid)                    0     0       0.6128       1.8691
  XGBoost (strong)                     0     0       0.9014       0.9287

Wald CF≠DML: 0/49 at 10%  →  linear CF adequate


### SHAP: which controls drive the XGBoost nuisance model at h=0 and h=12?

In [16]:
hs_shap = np.arange(HMAX+1)
fig, axes = plt.subplots(1, 2, figsize=(14,5))

frames_shap = []
for spec in valid_dyads:
    df_d = df_ext[[OUTCOME]+CTRL_FULL].copy()
    df_d['lpri_p'] = df_raw[spec['lpri_col']].reindex(df_ext.index)
    df_d['dyad']   = spec['code']
    for l in range(1,4): df_d[f'Ly{l}'] = df_d[OUTCOME].shift(l)
    frames_shap.append(df_d)
base_shap = pd.concat(frames_shap)
dums_shap = pd.get_dummies(base_shap['dyad'], prefix='D', drop_first=True).astype(float)
base_shap = pd.concat([base_shap.reset_index(drop=True), dums_shap.reset_index(drop=True)], axis=1)
X_shap    = [f'Ly{l}' for l in range(1,4)] + CTRL_FULL + dums_shap.columns.tolist()

for ax_idx, h in enumerate([0, 12]):
    base_shap['y_fwd'] = base_shap.groupby('dyad')[OUTCOME].transform(lambda s: s.shift(-h))
    sub = base_shap[['y_fwd']+X_shap].replace([np.inf,-np.inf],np.nan).dropna()
    if len(sub) < 50: axes[ax_idx].set_visible(False); continue
    xgb_s = get_xgb_stable()
    xgb_s.fit(sub[X_shap], sub['y_fwd'])
    exp = shap.TreeExplainer(xgb_s)
    sv  = exp.shap_values(sub[X_shap])
    imp = pd.Series(np.abs(sv).mean(axis=0), index=X_shap).sort_values(ascending=False).head(12)
    axes[ax_idx].barh(range(len(imp)), imp.values, color='steelblue', alpha=0.8)
    axes[ax_idx].set_yticks(range(len(imp)))
    axes[ax_idx].set_yticklabels(imp.index, fontsize=9)
    axes[ax_idx].invert_yaxis()
    axes[ax_idx].set_title(f'SHAP importance — l-model (E[WTI|X])\\nh={h}')
    axes[ax_idx].set_xlabel('Mean |SHAP|')
    axes[ax_idx].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(FIGURES/'Figure_10_shap.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_10_shap.png')

Saved: Figure_10_shap.png


### Figure: Full panel comparison — all estimators

In [17]:
hs = np.arange(HMAX+1)
fig, axes = plt.subplots(1, 2, figsize=(16,5))

ax = axes[0]
plot_items = []
for pname, irf_p in irf_ivw.items():
    sig = int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum())
    plot_items.append((f'IVW: {pname}', irf_p, sig))
for pname, irf_p in cf_panels.items():
    sig = int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum())
    plot_items.append((f'CF: {pname}', irf_p, sig))
for mname, irf_m in dml_results.items():
    sig = int((irf_m['lo90']>0).sum()+(irf_m['hi90']<0).sum())
    plot_items.append((f'DML: {mname}', irf_m, sig))

all_cols = plt.cm.Set2(np.linspace(0,1,len(plot_items)))
for (label, irf_p, sig), col in zip(plot_items, all_cols):
    ax.plot(hs, irf_p['coef'], color=col, lw=1.5, label=f'{label} ({sig}/49)')
    ax.fill_between(hs, irf_p['lo90'], irf_p['hi90'], color=col, alpha=0.06)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlim(0,HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
ax.set_xlabel('Horizon (months)'); ax.set_ylabel('Coefficient on log PRI')
ax.set_title('All pooled estimators — full comparison\\n(90% CI)')
ax.legend(fontsize=6.5); ax.grid(alpha=0.2)

# sig90 bar chart
ax2 = axes[1]
labels_all = [lab for lab,_,_ in plot_items]
sig90_all  = [sig for _,_,sig in plot_items]
bar_cols    = all_cols[:len(labels_all)]
ax2.barh(range(len(labels_all)), sig90_all, color=bar_cols, alpha=0.8)
ax2.set_yticks(range(len(labels_all)))
ax2.set_yticklabels(labels_all, fontsize=8)
ax2.axvline(4.9, color='black', lw=1, linestyle='--', label='~5% false positive')
ax2.set_xlabel('sig90 (out of 49)')
ax2.set_title('Significant horizons by estimator')
ax2.legend(fontsize=8); ax2.grid(alpha=0.2)

plt.suptitle('Full Panel Comparison: IVW vs CF vs DML', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_10_panel_all.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_10_panel_all.png')

Saved: Figure_10_panel_all.png


## 14. GDELT NLP sensitivity — full panel

CF panel re-run with GDELT controls added. We run this for both strong-only
and all-valid dyad sets.

In [18]:
cf_nlp = {}
if not NLP_COLS:
    print('NLP sensitivity skipped: no GDELT columns available (run NB05 addendum first).')
else:
    print(f'NLP sensitivity: CF panel + {len(NLP_COLS)} GDELT controls')
    print(f'  {NLP_COLS}')

    def run_cf_panel_nlp(spec_list, label='CF+NLP'):
        rows = []
        for h in range(HMAX+1):
            if h % 12 == 0: print(f'  h={h}...', end=' ', flush=True)
            frames = []
            for spec in spec_list:
                df_d = df_ext_nlp[[OUTCOME]+CTRL_FULL+NLP_COLS].copy()
                df_d['e']    = df_raw[spec['lpri_col']].reindex(df_ext_nlp.index)
                df_d['z']    = spec['best_series'].reindex(df_ext_nlp.index)
                df_d['dyad'] = spec['code']
                for l in range(1,4): df_d[f'Ly{l}'] = df_d[OUTCOME].shift(l)
                df_d['y_fwd'] = df_d[OUTCOME].shift(-h)
                frames.append(df_d)
            panel = pd.concat(frames).replace([np.inf,-np.inf],np.nan)
            lag_y  = [f'Ly{l}' for l in range(1,4)]
            exog_n = lag_y + CTRL_FULL + NLP_COLS
            panel  = panel[['y_fwd','e','z','dyad']+exog_n].dropna()
            if len(panel) < 60:
                rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)}); continue
            try:
                panel = panel.copy(); panel['cf_r'] = np.nan
                for d in panel['dyad'].unique():
                    mask = panel['dyad']==d
                    X_fs = add_constant(panel[mask][['z']+exog_n], has_constant='add')
                    panel.loc[mask,'cf_r'] = sm.OLS(panel[mask]['e'], X_fs).fit().resid
                panel = panel.dropna(subset=['cf_r'])
                dums  = pd.get_dummies(panel['dyad'], prefix='D', drop_first=True).astype(float)
                panel = pd.concat([panel.reset_index(drop=True),
                                   dums.reset_index(drop=True)], axis=1)
                X2 = add_constant(panel[['e','cf_r']+exog_n+dums.columns.tolist()], has_constant='add')
                fit2 = sm.OLS(panel['y_fwd'], X2).fit(cov_type='HC1')
                rows.append({'h':h,'coef':float(fit2.params.get('e',np.nan)),
                              'se':float(fit2.bse.get('e',np.nan)),'n':len(panel)})
            except Exception as ex:
                print(f'\n  [{label} h={h}] {type(ex).__name__}: {ex}')
                rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)})
        print(' done.')
        irf = pd.DataFrame(rows)
        irf['lo90'] = irf['coef']-1.645*irf['se']
        irf['hi90'] = irf['coef']+1.645*irf['se']
        return irf

    cf_nlp['All valid + NLP'] = run_cf_panel_nlp(valid_dyads)
    cf_nlp['All valid + NLP'].to_csv(RESULTS/'irf_cf_nlp_all.csv', index=False)
    if len(strong_dyads) >= 2:
        cf_nlp['Strong + NLP'] = run_cf_panel_nlp(strong_dyads)
        cf_nlp['Strong + NLP'].to_csv(RESULTS/'irf_cf_nlp_strong.csv', index=False)

    print()
    cf_ref_all = cf_panels.get('All valid (6)')
    for pname, irf_n in cf_nlp.items():
        sig_n  = int((irf_n['lo90']>0).sum()+(irf_n['hi90']<0).sum())
        base_key = 'Strong (US+JP)' if 'Strong' in pname else 'All valid (6)'
        irf_b  = cf_panels.get(base_key, pd.DataFrame())
        sig_b  = int((irf_b['lo90']>0).sum()+(irf_b['hi90']<0).sum()) if not irf_b.empty else 'N/A'
        print(f'  {pname:<30}  sig90={sig_n}/49  (vs core: {sig_b}/49)')

    print()
    print('Horizon-by-horizon sensitivity (all valid dyads):')
    if 'All valid + NLP' in cf_nlp and 'All valid (6)' in cf_panels:
        irf_core = cf_panels['All valid (6)']
        irf_nlp_ = cf_nlp['All valid + NLP']
        print(f'  {"h":>4}  {"Core":>10}  {"NLP":>10}  {"% change":>10}')
        for h in [0,6,12,24,36,48]:
            bc = float(irf_core.loc[h,'coef'])
            nc = float(irf_nlp_.loc[h,'coef'])
            pct = abs(nc-bc)/(abs(bc)+1e-10)*100 if not any(pd.isna([bc,nc])) else np.nan
            flag = '  ⚠' if (not pd.isna(pct) and pct > 50) else ''
            print(f'  {h:>4d}  {bc:>10.4f}  {nc:>10.4f}  {pct:>9.1f}%{flag}')

NLP sensitivity: CF panel + 4 GDELT controls
  ['gdelt_goldstein_mean', 'gdelt_sentiment_signal', 'gdelt_conflict_share', 'gdelt_coop_share']
  h=0...   h=12...   h=24...   h=36...   h=48...  done.
  h=0...   h=12...   h=24...   h=36...   h=48...  done.

  All valid + NLP                 sig90=0/49  (vs core: 0/49)
  Strong + NLP                    sig90=0/49  (vs core: 0/49)

Horizon-by-horizon sensitivity (all valid dyads):
     h        Core         NLP    % change
     0      0.0001      0.0001       35.1%
     6     -0.0015     -0.0018       19.9%
    12     -0.0002     -0.0007      279.4%  ⚠
    24      0.0001      0.0001        1.6%
    36      0.0007      0.0006       16.1%
    48      0.0014      0.0011       16.5%


## 15. GPR sensitivity

**GPR (Geopolitical Risk Index, Caldara-Iacoviello)** is a published alternative
measure of geopolitical risk. We test:
1. Does including GPR as a control change the US-China LP-IV result?
2. Can GPR itself serve as an alternative instrument?
3. How does the GPR-instrumented estimate compare to PRI-instrumented?

This provides a direct comparison between Saadaoui's PRI-based instrument
and the standard GPR measure.

In [19]:
gpr_cols = [c for c in df_ext.columns if 'gpr' in c.lower()]
print(f'GPR columns available: {gpr_cols}')

if not gpr_cols:
    print('GPR columns not found in df_extended — GPR sensitivity skipped.')
    gpr_results = {}
else:
    gpr_results = {}
    endog_us = df_raw[us_spec['lpri_col']].reindex(df_ext.index) if us_spec else None
    instr_us = us_spec['best_series'].reindex(df_ext.index) if us_spec else None

    if us_spec is not None:
        print('Running GPR sensitivity for US-China...')

        # 1. LP-IV with GPR excluded from controls
        ctrl_no_gpr = [c for c in CTRL_FULL if c not in gpr_cols]
        irf_no_gpr  = lp_iv(df_ext, endog_us, instr_us, ctrl_no_gpr,
                             include_endog_lags=us_spec['can_elags'], label='no-GPR')
        gpr_results['IV, no GPR controls'] = irf_no_gpr

        # 2. LP-IV baseline (with GPR controls — already in CTRL_FULL)
        gpr_results['IV, with GPR controls'] = irf_iv['us']

        # 3. GPR as instrument (does GPR instrument the PRI→WTI channel?)
        for gcol in gpr_cols:
            gpr_s = df_ext[gcol]
            # Use differenced GPR as instrument candidate
            dgpr  = gpr_s.diff()
            F_gpr = first_stage_F(endog_us, dgpr, df_ext[CTRL_CORE], endog_lags=2)
            print(f'  GPR instrument ({gcol}): F={F_gpr:.1f}')
            if not pd.isna(F_gpr) and F_gpr >= MIN_F:
                irf_gpr_iv = lp_iv(df_ext, endog_us, dgpr, CTRL_CORE,
                                    include_endog_lags=True, label=f'GPR-IV {gcol}')
                gpr_results[f'GPR-IV ({gcol})'] = irf_gpr_iv
                sig_gpr = int((irf_gpr_iv['lo90']>0).sum()+(irf_gpr_iv['hi90']<0).sum())
                print(f'    sig90={sig_gpr}/49')

        # 4. Reduced form: GPR → WTI (GPR as predictor of WTI directly)
        for gcol in gpr_cols:
            irf_gpr_rf = lp_reduced_form(df_ext, df_ext[gcol], CTRL_CORE, label=f'GPR RF {gcol}')
            gpr_results[f'GPR reduced form ({gcol})'] = irf_gpr_rf
            sig_rf = int((irf_gpr_rf['lo90']>0).sum()+(irf_gpr_rf['hi90']<0).sum())
            print(f'  GPR reduced form ({gcol}): sig90={sig_rf}/49')

        # Summary
        print()
        print('GPR SENSITIVITY SUMMARY')
        print(f'  {"Specification":<35} {"sig90":>7} {"β(h=12)":>10}')
        for rname, irf_r in gpr_results.items():
            sig   = int((irf_r['lo90']>0).sum()+(irf_r['hi90']<0).sum())
            h12   = float(irf_r.loc[irf_r['h']==12,'coef'].values[0]) if 12 < len(irf_r) else np.nan
            print(f'  {rname:<35} {sig:>7} {h12:>10.4f}')

        # Figure
        hs = np.arange(HMAX+1)
        fig, ax = plt.subplots(figsize=(12, 5))
        GPR_COLS = plt.cm.tab10(np.linspace(0,1,len(gpr_results)))
        for (rname, irf_r), col in zip(gpr_results.items(), GPR_COLS):
            sig = int((irf_r['lo90']>0).sum()+(irf_r['hi90']<0).sum())
            ax.plot(hs, irf_r['coef'], color=col, lw=1.8, label=f'{rname} ({sig}/49)')
            ax.fill_between(hs, irf_r['lo90'], irf_r['hi90'], color=col, alpha=0.08)
        ax.axhline(0, color='black', lw=0.8)
        ax.set_xlim(0,HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
        ax.set_xlabel('Horizon (months)'); ax.set_ylabel('Coefficient')
        ax.set_title('GPR Sensitivity: PRI instrument vs GPR instrument vs GPR controls\\n(US-China, 90% CI)')
        ax.legend(fontsize=8); ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.savefig(FIGURES/'Figure_10_gpr_sensitivity.png', dpi=300, bbox_inches='tight')
        plt.close()
        print('Saved: Figure_10_gpr_sensitivity.png')

GPR columns available: ['gpr_chn_l1', 'gpr_usa_l1']
Running GPR sensitivity for US-China...
  GPR instrument (gpr_chn_l1): F=1.8
  GPR instrument (gpr_usa_l1): F=0.5
  GPR reduced form (gpr_chn_l1): sig90=3/49
  GPR reduced form (gpr_usa_l1): sig90=41/49

GPR SENSITIVITY SUMMARY
  Specification                         sig90    β(h=12)
  IV, no GPR controls                      17    -0.0233
  IV, with GPR controls                    16    -0.0229
  GPR reduced form (gpr_chn_l1)             3    -0.0261
  GPR reduced form (gpr_usa_l1)            41     0.0160
Saved: Figure_10_gpr_sensitivity.png


## 16. Structural heterogeneity — multiple sub-sample splits

We test three different sub-sample splits for US-China:
1. **Crisis vs non-crisis** (GFC + China crash)
2. **Pre- vs post-2008** (GFC as break)
3. **Pre- vs post-2015** (China crash / commodity supercycle end as break)

For each split we report: sig90, h=12 coefficient, and whether IV > OLS
(endogeneity direction). No Wald test between sub-samples because crisis windows
are underpowered — reported as descriptive evidence only.

In [20]:
if us_spec is None:
    print('Heterogeneity skipped: US-China not valid.')
    het_results = {}
else:
    endog_us = df_raw[us_spec['lpri_col']].reindex(df_ext.index)
    instr_us = us_spec['best_series'].reindex(df_ext.index)

    splits = {
        'GFC crisis (2007-09)':       (df_ext.index >= '2007-01-01') & (df_ext.index <= '2009-12-31'),
        'China crash (2015-16)':      (df_ext.index >= '2015-01-01') & (df_ext.index <= '2016-12-31'),
        'Combined crisis':            ((df_ext.index >= '2007-01-01') & (df_ext.index <= '2009-12-31')) |
                                      ((df_ext.index >= '2015-01-01') & (df_ext.index <= '2016-12-31')),
        'Non-crisis':                 ~(((df_ext.index >= '2007-01-01') & (df_ext.index <= '2009-12-31')) |
                                        ((df_ext.index >= '2015-01-01') & (df_ext.index <= '2016-12-31'))),
        'Pre-2008':                   df_ext.index < '2008-01-01',
        'Post-2008':                  df_ext.index >= '2008-01-01',
        'Pre-2015':                   df_ext.index < '2015-01-01',
        'Post-2015':                  df_ext.index >= '2015-01-01',
    }

    het_results = {}
    print('STRUCTURAL HETEROGENEITY — US-China LP-IV')
    print('='*75)
    print(f'  {"Sub-sample":<30} {"n":>5} {"sig90":>7} {"β(h=6)":>10} {"β(h=12)":>10}  Note')
    print('-'*75)

    for sname, mask in splits.items():
        n_obs = mask.sum()
        if n_obs < 50:
            print(f'  {sname:<30} {n_obs:>5}  SKIPPED (n<50)')
            continue
        irf_s = lp_iv(df_ext[mask], endog_us[mask], instr_us[mask], CTRL_FULL,
                       include_endog_lags=us_spec['can_elags'], label=sname)
        het_results[sname] = irf_s
        irf_s.to_csv(RESULTS/f'irf_het_{sname[:15].replace(" ","_")}.csv', index=False)

        sig  = int((irf_s['lo90']>0).sum()+(irf_s['hi90']<0).sum())
        h6   = float(irf_s.loc[irf_s['h']==6,'coef'].values[0])  if 6  < len(irf_s) else np.nan
        h12  = float(irf_s.loc[irf_s['h']==12,'coef'].values[0]) if 12 < len(irf_s) else np.nan
        Fmin = round(float(irf_s['F'].min()), 1)
        note = f'F_min={Fmin}'
        print(f'  {sname:<30} {n_obs:>5} {sig:>7} {h6:>10.4f} {h12:>10.4f}  {note}')

    # Figure: 4-panel comparison
    if len(het_results) >= 4:
        fig, axes = plt.subplots(2, 4, figsize=(24, 9), squeeze=False)
        hs = np.arange(HMAX+1)
        for ax_i, (sname, irf_s) in enumerate(list(het_results.items())[:8]):
            ax = axes[ax_i//4][ax_i%4]
            sig = int((irf_s['lo90']>0).sum()+(irf_s['hi90']<0).sum())
            n   = splits[sname].sum()
            ax.plot(hs, irf_s['coef'], color='steelblue', lw=2, label=f'LP-IV (sig={sig})')
            ax.fill_between(hs, irf_s['lo90'], irf_s['hi90'], color='steelblue', alpha=0.18)
            ax.axhline(0, color='black', lw=0.8)
            ax.set_xlim(0,HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
            ax.set_title(f'{sname}\\nn={n}, sig90={sig}/49', fontsize=9)
            ax.set_xlabel('Horizon (months)', fontsize=8)
            ax.grid(alpha=0.2)
        plt.suptitle('US-China LP-IV: Structural Heterogeneity\\n'
                     'Multiple sub-sample splits — descriptive (underpowered for short windows)', fontsize=11)
        plt.tight_layout()
        plt.savefig(FIGURES/'Figure_10_heterogeneity.png', dpi=300, bbox_inches='tight')
        plt.close()
        print('Saved: Figure_10_heterogeneity.png')

STRUCTURAL HETEROGENEITY — US-China LP-IV
  Sub-sample                         n   sig90     β(h=6)    β(h=12)  Note
---------------------------------------------------------------------------
  GFC crisis (2007-09)              36  SKIPPED (n<50)
  China crash (2015-16)             24  SKIPPED (n<50)
  Combined crisis                   60       1    -0.0362    -0.0473  F_min=218.7
  Non-crisis                       325      15    -0.0728     0.0326  F_min=33.5
  Pre-2008                         215      23    -0.1948    -0.0780  F_min=292.3
  Post-2008                        170       8    -0.2948    -0.2165  F_min=115.3
  Pre-2015                         299      26    -0.1727    -0.0868  F_min=433.6
  Post-2015                         86       5    -0.1774    -0.2084  F_min=40.6
Saved: Figure_10_heterogeneity.png


## 17. Power analysis — all sub-samples and panel configurations

In [21]:
alpha_c = stats.norm.ppf(0.90)
beta_c  = stats.norm.ppf(0.80)

us_se   = irf_iv['us']['se'].median() if 'us' in irf_iv else 0.15
se_diff = np.sqrt(2) * us_se

print('POWER ANALYSIS — MINIMUM DETECTABLE EFFECT AT 80% POWER')
print('='*70)
print(f'US-China median SE: {us_se:.4f}  |  SE of difference: {se_diff:.4f}')
print()
print(f'  {"Sub-sample/scenario":<35} {"n":>6} {"MDE @ 80%":>12}')
print('  ' + '-'*55)

power_scenarios = {
    'GFC crisis only':              36,
    'China crash only':             24,
    'Combined crisis (60 months)':  60,
    'VIX high regime (NB07)':       185,
    'Non-crisis':                   325,
    'US-China only (full)':         385,
    f'Panel all valid ({len(valid_dyads)}×385)': len(valid_dyads)*385,
    f'Panel strong ({len(strong_dyads)}×385)':   len(strong_dyads)*385,
}
for sname, n in power_scenarios.items():
    se_s = se_diff * np.sqrt(185/max(n,1))
    mde  = (alpha_c + beta_c) * se_s
    print(f'  {sname:<35} {n:>6} {mde:>12.4f}')

print()
print('Regime Wald test (NB07) needs Δ ≥ '
      f'{(alpha_c+beta_c)*se_diff*np.sqrt(185/185):.3f} for 80% power at n=185.')

delta_g = np.linspace(0, 0.5, 200)
fig, ax = plt.subplots(figsize=(10,5))
for sname, n in list(power_scenarios.items())[:6]:
    SE = se_diff * np.sqrt(185/max(n,1))
    pw = [1-stats.norm.cdf(alpha_c - d/SE) for d in delta_g]
    ax.plot(delta_g, pw, lw=2, label=f'{sname} (n={n})')
ax.axhline(0.80, color='black', lw=1, linestyle='--')
ax.set_xlabel('True effect Δ'); ax.set_ylabel('Power')
ax.set_title('Power curves — regime and sub-sample tests')
ax.legend(fontsize=8); ax.set_xlim(0,0.5); ax.set_ylim(0,1); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_10_power.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_10_power.png')

POWER ANALYSIS — MINIMUM DETECTABLE EFFECT AT 80% POWER
US-China median SE: 0.0860  |  SE of difference: 0.1217

  Sub-sample/scenario                      n    MDE @ 80%
  -------------------------------------------------------
  GFC crisis only                         36       0.5857
  China crash only                        24       0.7173
  Combined crisis (60 months)             60       0.4537
  VIX high regime (NB07)                 185       0.2584
  Non-crisis                             325       0.1949
  US-China only (full)                   385       0.1791
  Panel all valid (6×385)               2310       0.0731
  Panel strong (3×385)                  1155       0.1034

Regime Wald test (NB07) needs Δ ≥ 0.258 for 80% power at n=185.
Saved: Figure_10_power.png


## 18. Final honest summary

In [22]:
print()
print('='*95)
print('NOTEBOOK 10 — COMPLETE RESULTS SUMMARY')
print('='*95)

# A. Instrument
print('\nA. INSTRUMENT DIAGNOSTIC (best instrument under full control set)')
print(f'  {"Dyad":<22} {"Best instr":>12} {"F_diag":>8} {"F_min":>8} {"exog_p":>8}  Status')
for r in diag_rows:
    F_s = f'{r["best_F"]:.1f}' if not pd.isna(r['best_F']) else ' NaN'
    p_s = f'{r["exog_p"]:.3f}' if not pd.isna(r['exog_p']) else ' NaN'
    status = 'STRONG' if r['strong'] else ('valid' if r['valid'] else 'weak')
    print(f'  {r["name"]:<22} {str(r["best_name"]):>12} {F_s:>8} {"---":>8} {p_s:>8}  {status}')
print(f'  Strong (F≥{STRONG_F:.0f}): {len(strong_dyads)}/12   Valid (F≥{MIN_F:.0f}): {len(valid_dyads)}/12')

# B. Dyad LP-IV
print('\nB. DYAD-BY-DYAD: LP-IV vs OLS vs REDUCED FORM')
print(summ_df[['name','instrument','F_diag','F_min_lp','sig_iv','sig_ols','sig_rf']].to_string(index=False))

# C. Specification curve US-China
print('\nC. SPECIFICATION CURVE — US-CHINA')
for sname, irf_s in spec_curve.items():
    sig  = int((irf_s['lo90']>0).sum()+(irf_s['hi90']<0).sum())
    h12  = float(irf_s.loc[irf_s['h']==12,'coef'].values[0])
    print(f'  {sname:<22}  sig90={sig:2d}/49  β(h=12)={h12:.4f}')

# D. Placebo
if 'perm_arr' in dir():
    print(f'\nD. PLACEBO TEST: p={p_val:.3f}  rank={rank_pct:.0f}th percentile')
    print(f'   Observed sig90={sig_obs}  |  Permutation 95th pct={np.percentile(perm_arr,95):.0f}')

# E. H2
print('\nE. EXTERNAL VALIDITY (H2: US == Japan)')
if sig_h2 is not None:
    v = 'confirmed' if sig_h2 <= 5 else 'REJECTED'
    print(f'  Wald: {sig_h2}/49 → {v}')
    print(f'  Note: Japan sig90={jp_sig}/49 despite F={next(r["best_F"] for r in valid_dyads if r["code"]=="jp"):.0f}')
    if jp_sig == 0:
        print('  Japan-China null is a GENUINE FINDING, not a power failure.')

# F. Meta-analytic pooling
print('\nF. META-ANALYTIC IVW POOLING')
for pname, irf_p in irf_ivw.items():
    sig  = int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum())
    Qsig = int((irf_p['Q_p']<ALPHA).sum())
    mI2  = irf_p['I2'].mean()
    print(f'  {pname:<38}  sig90={sig:2d}/49  Q_sig={Qsig}/49  I²={mI2:.0f}%')

# G. CF panel
print('\nG. CF POOLED PANEL')
for pname, irf_p in cf_panels.items():
    sig = int((irf_p['lo90']>0).sum()+(irf_p['hi90']<0).sum())
    n   = irf_p['n'].mean()
    print(f'  {pname:<30}  sig90={sig:2d}/49  n_mean={n:.0f}')

# H. DML
print('\nH. PANEL DML-PLIV')
for mname, irf_m in dml_results.items():
    sig  = int((irf_m['lo90']>0).sum()+(irf_m['hi90']<0).sum())
    mse  = irf_m['se'].median()
    flag = '  ⚠ unstable (median SE > 1)' if mse > 1.0 else ''
    print(f'  {mname:<30}  sig90={sig:2d}/49  median_SE={mse:.4f}{flag}')
if sig_wd is not None:
    print(f'  Wald CF≠DML: {sig_wd}/49')

# I. NLP sensitivity
print('\nI. GDELT NLP SENSITIVITY')
if cf_nlp:
    for pname, irf_n in cf_nlp.items():
        sig = int((irf_n['lo90']>0).sum()+(irf_n['hi90']<0).sum())
        print(f'  {pname:<30}  sig90={sig:2d}/49')
else:
    print('  Not run — GDELT data not available')

# J. GPR sensitivity
print('\nJ. GPR SENSITIVITY')
for rname, irf_r in gpr_results.items():
    sig  = int((irf_r['lo90']>0).sum()+(irf_r['hi90']<0).sum())
    h12  = float(irf_r.loc[irf_r['h']==12,'coef'].values[0]) if 12 < len(irf_r) else np.nan
    print(f'  {rname:<38}  sig90={sig:2d}/49  β(h=12)={h12:.4f}')

# K. Heterogeneity
print('\nK. STRUCTURAL HETEROGENEITY (US-China)')
for sname, irf_s in het_results.items():
    sig = int((irf_s['lo90']>0).sum()+(irf_s['hi90']<0).sum())
    n   = splits[sname].sum()
    h12 = float(irf_s.loc[irf_s['h']==12,'coef'].values[0]) if 12 < len(irf_s) else np.nan
    print(f'  {sname:<30}  n={n:>4}  sig90={sig:2d}/49  β(h=12)={h12:.4f}')

# L. Power
print('\nL. POWER — MDE AT 80% POWER')
mde_185 = (alpha_c+beta_c)*se_diff
print(f'  NB07 Wald (n=185/regime): MDE={mde_185:.4f}')
print(f'  → 0/49 is consistent with true effect Δ < {mde_185:.3f}')
print(f'  Cannot distinguish "no regime effect" from "effect below {mde_185:.3f}"')

print()
print('All figures saved to:', FIGURES)
print('All CSVs saved to   :', RESULTS)
print('='*95)


NOTEBOOK 10 — COMPLETE RESULTS SUMMARY

A. INSTRUMENT DIAGNOSTIC (best instrument under full control set)
  Dyad                     Best instr   F_diag    F_min   exog_p  Status
  US–China                      d2pri    242.7      ---    0.125  STRONG
  Japan–China                   d2pri    112.8      ---    0.723  STRONG
  Australia–China             L2dlpri     60.5      ---    0.052  STRONG
  S.Korea–China               L1dlpri      0.5      ---    0.770  weak
  France–China                L2dlpri     11.5      ---    0.605  valid
  Germany–China               L1dlpri     11.9      ---    0.533  valid
  India–China                 L2dlpri      1.2      ---    0.595  weak
  Indonesia–China             L2dlpri      6.2      ---    0.098  weak
  Pakistan–China              L2dlpri      2.2      ---    0.253  weak
  Russia–China                L2dlpri     26.9      ---    0.268  valid
  Vietnam–China               L2dlpri      0.7      ---    0.063  weak
  UK–China                    